A kaggle username and its PAT is needed to retrieve the data, get yours from https://www.kaggle.com/settings/api then generate a "Legacy API Credentials".
This data goes within the `.env` file.

Before running this notebook, initialize a virtual environment and install requirements:
1. `python -m venv .venv`: to create the virtual environment.
2. `.venv\scripts\activate`: to activate the virtual environment using Windows.
3. `pip install -r requirements.txt`: to install the list of requirements.

In [ ]:
import kaggle
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

Download and read file (requires Kaggle account):

In [ ]:
datasets_path = Path("dataset")
kaggle.api.dataset_download_files(
    "dgomonov/new-york-city-airbnb-open-data", path=datasets_path, unzip=True
)
data = pd.read_csv(datasets_path / "AB_NYC_2019.csv")

To understanding what kind of and how much data we have:

In [ ]:
# The field name of data
print("The field name of data: ", data.columns)

# Number of fields in data
print("Number of fields in data: ", len(data.columns))

# Number of data in data
print("Number of data in data: ", len(data))

In [ ]:
# data output
data

In [ ]:
# View information such as data field properties and count
# For find out how much of datas were null
data.info()

In [ ]:
# Check the number of nulls in each field
data.isnull().sum()

I check some datas which were null in last_review and reviews_per_month, and in some of them, the number_of_reviews was 0. So it's obvious that when number_of_reviews was 0 these two columns will be zero. therefore I will put zero on null column. 

In [ ]:
data.loc[data.number_of_reviews == 0, "reviews_per_month"] = 0
data.loc[data.number_of_reviews == 0, "last_review"] = 0

In [ ]:
data

Without name and host_name data will be useless so remove them

In [ ]:
# Removing null values in name and host_name fields
data = data[pd.notnull(data["name"])]
data = data[pd.notnull(data["host_name"])]

So now we check how much datas were null

In [ ]:
# Recheck the number of nulls in each field
data.isnull().sum()

As you see we don't have anyn null data.
For next step I gonna findout we have any outliers or not. So we're gonna sort data.

In [ ]:
data.sort_values(by=["latitude"])

so as you see latitude datas very close so we don't have any outliers on it. 

In [ ]:
data.sort_values(by=["longitude"])

longitude osn't have any outliers too.

In [ ]:
data.sort_values(by=["price"])

but we have big interval. So I'm curios to see what is the mean of these datas.

In [ ]:
np.mean(data.price)

So the price is not too big and maybe the datas that have big price is outliers. I'm gonna findout is really big price is outliers or not so I draw a chart only for price. 

In [ ]:
import matplotlib.pyplot as plt

plt.hist(data.price, bins=50)

So as you see more than 2000. maybe will be outliers. therefore I'm gonna check how musch datas more than 2000.

In [ ]:
len(data[data.price > 2000])

In 48858 datas we have only 86 datas more than 2000. So I think, it's obvious that these data were outliers. So I remove them and after that I will plot again to findout the exact price were begining outliers datas.

In [ ]:
data = data[data.price < 2000]

In [ ]:
plt.hist(data.price, bins=50)

In [ ]:
len(data[data.price > 1000])

So more than 1000 is outliers and I gonna delete them

In [ ]:
data = data[data.price <= 1000]

In [ ]:
plt.hist(data.price, bins=50)

Now we can see we have compact data in price. So go to sort another column to findout the outliers if it has.

In [ ]:
data.sort_values(by=["minimum_nights"])

Maybe we have outliers. Let's check it.

In [ ]:
plt.hist(data.minimum_nights, bins=50)

In [ ]:
len(data[data.minimum_nights > 200])

Obviously more than 200 nigths is outliers.

In [ ]:
data = data[data.minimum_nights < 200]

In [ ]:
plt.hist(data.minimum_nights, bins=50)

In [ ]:
len(data[data.minimum_nights > 100])

We save more than 100 nights because it's not outliers anymore. check other column. 

In [ ]:
data.sort_values(by=["number_of_reviews"])

In [ ]:
plt.hist(data.number_of_reviews, bins=50)

In [ ]:
len(data[data.number_of_reviews > 300])

In [ ]:
len(data[data.number_of_reviews > 400])

more than 400 were outliers.

In [ ]:
data = data[data.number_of_reviews <= 400]

We ignore reviews_per_month because many of them put by myself. 

In [ ]:
data.sort_values(by=["calculated_host_listings_count"])

In [ ]:
plt.hist(data.calculated_host_listings_count, bins=50)

In this case we don't have any outliers because even in more than 300 we have some datas that were shown in this plot.

In [ ]:
data.sort_values(by=["availability_365"])

In [ ]:
plt.hist(data.availability_365, bins=50)

This is completely compact data and it's very good feature to predict datas.

In [ ]:
len(data)

**So now I have 48511 compact data that I can ask question and answer by these datas.**

1. Which neighborhood has the most hosts?

In [ ]:
a = data.groupby(by=["neighbourhood"]).neighbourhood.count()
a = a.sort_values(ascending=False)
print(a)

2. Which neighborhood has the most expensive?

In [ ]:
a = data.groupby(by=["neighbourhood"]).price.mean()
a = a.sort_values(ascending=False)
print(a)

3. Which room_type Airbnb has?

In [ ]:
a = data.groupby(by=["room_type"]).room_type.count()
a = a.sort_values(ascending=False)
print(a)

4. Which room_type have more reviews?

In [ ]:
a = data.groupby(by=["room_type"]).number_of_reviews.sum()
a = a.sort_values(ascending=False)
print(a)

5. Which hosts has more cabin?

In [ ]:
a = data.groupby(by=["host_name"]).host_name.count()
a = a.sort_values(ascending=False)
print(a)

6. Which host has the cheapest cabins?

In [ ]:
a = data.groupby(by=["host_name"]).price.mean()
a = a.sort_values(ascending=True)
print(a)

7. Which neighbourhood_group is the biggest one?

In [ ]:
a = data.groupby(by=["neighbourhood_group"]).neighbourhood_group.count()
a = a.sort_values(ascending=False)
print(a)

8. Which neighbourhood_group is the most expensive?

In [ ]:
a = data.groupby(by=["neighbourhood_group"]).price.mean()
a = a.sort_values(ascending=False)
print(a)

9. Which neighbourhood_group has the most possibility to available in year?

In [ ]:
a = data.groupby(by=["neighbourhood_group"]).availability_365.sum()
a = a.sort_values(ascending=False)
print(a)

10. Which neighbourhood has the most possibility to available in year?

In [ ]:
a = data.groupby(by=["neighbourhood"]).availability_365.sum()
a = a.sort_values(ascending=False)
print(a)

11. Which neigbourhood_group has the best hosts to stay for a few nights?

In [ ]:
a = data.groupby(by=["neighbourhood_group"]).minimum_nights.mean()
a = a.sort_values(ascending=True)
print(a)

11. Which neigbourhood has the best hosts to stay for a few nights?

In [ ]:
a = data.groupby(by=["neighbourhood"]).minimum_nights.mean()
a = a.sort_values(ascending=True)
print(a)

12. Which host_name is the most popular hosts between customers?

In [ ]:
a = data.groupby(by=["host_name"]).calculated_host_listings_count.mean()
a = a.sort_values(ascending=False)
print(a)

I uses first lines of https://www.kaggle.com/dineshkumaranbalagan/nyc-airbnb-deep-data-geospatial-analysis to has a better code.